# Script to create datasets based on Nuggets scores, finetune, and eval them

Meher Mankikar
Summer 2024 Internship

In [3]:
import pandas as pd
import requests
import aiobotocore.session
import s3fs
import json
import os
import random

from datasets import Dataset, DatasetDict, load_from_disk, load_dataset
from requests.auth import HTTPBasicAuth
from IPython.display import JSON

/mnt/efs/mehermankikar/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
USER_ID = '65b91e6cf11e85c6770bc712'
basic = HTTPBasicAuth(USER_ID, '')
HOST = "https://ml-orchestration.ml-internal.scale.com"

In [5]:
def get_s3fs():
    s3_session = aiobotocore.session.AioSession(profile="ml-worker")
    storage_options = {"session": s3_session}
    fs = s3fs.S3FileSystem(**storage_options)
    return fs
def save_ds_s3(ds, path: str):
    fs = get_s3fs()
    ds.save_to_disk(path, storage_options=fs.storage_options)
def load_ds_s3(path: str):
    fs = get_s3fs()
    dataset = load_from_disk(path, storage_options=fs.storage_options)
    return dataset

## Save the results of e2e generation to csvs with task_ids

### Todo: In order to use this code, update to use the JSON file that was the result of evaluate_vllm_generation. Also update the csv_file_path

In [4]:
import json
import csv

# Open the results file and read json data
with open("/mnt/efs/mehermankikar/bigcode-evaluation-harness/bigcode_eval/tasks/ots_example_reruns_0810/debugging_dataset_rerun_results_0811.json", "r") as f:
    data = json.load(f)

# read the s3 dataset
e2e_run_1_ds = "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset/dataset/"
e2e_ds = load_ds_s3(e2e_run_1_ds)

# save json data and task_ids from s3 dataset to a csv file 
# Create a list to store the rows of the CSV
csv_rows = []

# Iterate over each row in the JSON data
i = 0
for row in e2e_ds["train"]:
    pass_1 = data[str(i)][0]['pass@1']
    
    # Get the task_id from the dataset
    task_id = row["task_id"]
    
    # Create a new row with the three values
    csv_row = [i, task_id, pass_1]
    
    # Append the row to the list
    csv_rows.append(csv_row)
    i+=1

# Define the path of the CSV file
csv_file_path = "/mnt/efs/mehermankikar/bigcode-evaluation-harness/bigcode_eval/tasks/ots_example_reruns_0810/debugging_dataset_rerun_results_0811.csv"

# Open the CSV file in write mode
with open(csv_file_path, "w", newline="") as csv_file:
    # Create a CSV writer object
    csv_writer = csv.writer(csv_file)
    
    # Write the header row
    csv_writer.writerow(["Index", "Task ID", "Pass@1"])
    
    # Write the data rows
    csv_writer.writerows(csv_rows)

# Print a message to confirm the CSV file has been saved
print(f"CSV file saved at {csv_file_path}")

CSV file saved at /mnt/efs/mehermankikar/bigcode-evaluation-harness/bigcode_eval/tasks/ots_example_reruns_0810/debugging_dataset_rerun_results_0811.csv


## Go through the csv results and sort by pass@1 scores. Save as two separate hf datasets to s3

### Todo: udpate the csv file path to read from what you wrote above

In [6]:
import csv

# read the data from the two csv files 
def read_csv(path):
    csv_data = []
    with open(path, 'r') as file:
        csv_reader = csv.reader(file)
        for row in csv_reader:
            csv_data.append(row)
    return csv_data

# Sort the data
# combined_data = first_half + second_half
combined_data = read_csv("/mnt/efs/mehermankikar/bigcode-evaluation-harness/bigcode_eval/tasks/ots_example_reruns_0810/debugging_dataset_rerun_results_0811.csv")[1:]
sorted_data = sorted(combined_data, key=lambda x: float(x[2])) 
print(len(sorted_data))

# These were used to filter out tasks that were too long. 
code_llama_ids_to_remove = [str(i) for i in [17, 20, 21, 23, 24, 25, 26, 27, 45, 50, 53, 66, 82, 188, 513]]
llama_3_ids_to_remove = [str(i) for i in [15, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 34, 37, 45, 47, 48, 50, 51, 53, 56, 66, 70, 74, 75, 80, 82, 84, 86, 91, 92,188, 190, 508, 513]]
llama_3_ids_to_remove_test2 = [str(i) for i in [17]]


sorted_data = [x for x in sorted_data if x[0] not in llama_3_ids_to_remove]
print(len(sorted_data))

half_len = len(sorted_data) // 2
ten_percent = len(sorted_data) // 10
thirty_percent = len(sorted_data) // 10 * 3
bottom_50 = sorted_data[:half_len]
top_50 = sorted_data[half_len:]
bottom_10 = sorted_data[:ten_percent]
top_10 = sorted_data[len(sorted_data) - ten_percent:]
bottom_30 = sorted_data[:thirty_percent]
top_30 = sorted_data[len(sorted_data) - thirty_percent:]


519
485


In [7]:
len(sorted_data), len(bottom_50), len(top_50)

(485, 242, 243)

In [8]:
len(bottom_50), len(top_50), len(bottom_10), len(top_10), len(bottom_30), len(top_30)

(242, 243, 48, 48, 144, 144)

In [10]:
bottom_30[:10]

[['22', '661a1145f4dc49a1d940a867', '0.5426829268292683'],
 ['307', '665f861c1cacac11e7c551b5', '0.5426829268292683'],
 ['60', '661a114a0b04a75f5dba0bf5', '0.5487804878048781'],
 ['483', '66033ef8c2805b0da320f75f', '0.5487804878048781'],
 ['95', '660b439a4f6501550ff1a984', '0.5548780487804879'],
 ['345', '665f86591cacac11e7c5632d', '0.5548780487804879'],
 ['79', '66033ef543d4dd2278001859', '0.5609756097560976'],
 ['89', '660b01b83cbcdfee24328eb5', '0.5609756097560976'],
 ['111', '666371f3f20066166a3364d9', '0.5609756097560976'],
 ['210', '665f86de1cacac11e7c5866f', '0.5609756097560976']]

## Save as two new hf datasetdicts -- need to get all the stuff from the original hf datasets

In [11]:
e2e_run_full_ds = "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset/dataset/"
e2e_ds = load_ds_s3(e2e_run_full_ds)

def get_list_with_data(list_without_data):
    list_w_data = []
    for elm in list_without_data:
        id, task_id, pass_1 = elm

        dataset_to_use = e2e_ds
        id = int(id)

        hf_dataset_element = dataset_to_use["train"].select([id])

        messages = hf_dataset_element["messages"][0]
        # task_attempt_ids = hf_dataset_element["task_attempt_ids"][0]

        new_data = [messages, task_id]#, task_attempt_ids]

        # check messages is same type
        assert type(dataset_to_use["train"][id]["messages"]) == type(messages)

        # check task_id is same type
        assert type(dataset_to_use["train"][id]["task_id"]) == type(task_id)

        list_w_data.append(new_data)

    return list_w_data

bottom_50_with_data = get_list_with_data(bottom_50)
top_50_with_data = get_list_with_data(top_50)
bottom_10_with_data = get_list_with_data(bottom_10)
top_10_with_data = get_list_with_data(top_10)
bottom_30_with_data = get_list_with_data(bottom_30)
top_30_with_data = get_list_with_data(top_30)

In [11]:
top_30_with_data[-1][0]

[{'content': 'Write a Python function to assess whether the reverse of the first string matches the second string regardless of case sensitivity.',
  'context': None,
  'role': 'user'},
 {'content': 'def is_reverse_match(a, b):\n    return a.lower()[::-1] == b.lower()',
  'context': {'tests': [['assert is_reverse_match("man", "Nam")',
     'assert is_reverse_match("Hello", "olleh")',
     'assert not is_reverse_match("name", "man")']]},
  'role': 'assistant'}]

In [21]:
test_low_50 = load_ds_s3("s3://scale-ml/users/mehermankikar/nuggets_datasets/llama-3-instruct/e2e_llama-3-instruct-600_high-50_nuggets/dataset/")
test_low_50["train"]["task_id"]
for i in range(len(bottom_50_with_data)):
    assert top_50_with_data[i][1] in test_low_50["train"]["task_id"]

In [19]:
len(bottom_30_with_data), len(top_30_with_data)

(153, 153)

In [12]:
import pandas as pd
from datasets import DatasetDict, Dataset
from sklearn.model_selection import train_test_split

def create_dataset_dict(list_with_data, isBottom, percent):
    # This is only the train data now. Get validation from original full ds
    df1 = pd.DataFrame(list_with_data, columns=['messages', 'task_id'])#, 'task_attempt_ids'])
    train_df = df1

    # train_df, val_df = train_test_split(df1, test_size=0.1, random_state=42)
    nuggets_full_600 = load_ds_s3("s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset/dataset/")
    nuggets_full_600_val = nuggets_full_600["eval"]
    val_df = nuggets_full_600_val.data.to_pandas()

    if isBottom:
        val_df = val_df[:percent]
    else:
        val_df = val_df[-percent:]

    # Convert train_df and val_df back to datasets
    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    val_dataset = Dataset.from_pandas(val_df, preserve_index=False)


    return DatasetDict({"train": train_dataset, "validation": val_dataset})



In [13]:
first_half_dict = create_dataset_dict(bottom_50_with_data, True, 50)
second_half_dict = create_dataset_dict(top_50_with_data, False, 50)

In [14]:
# Combine first_half_dict and second_half_dict and save them as the full dataset
# This may be different from the original dataset if we filtered out any tasks that were too long. 
full_train_pd = pd.concat([first_half_dict["train"].to_pandas(), second_half_dict["train"].to_pandas()])
full_val_pd = pd.concat([first_half_dict["validation"].to_pandas(), second_half_dict["validation"].to_pandas()])
full_dataset_dict = DatasetDict({"train": Dataset.from_pandas(full_train_pd), "validation": Dataset.from_pandas(full_val_pd)})
save_ds_s3(full_dataset_dict, "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset_redo_0811/llama-3-instruct/combined_ds/dataset/")


Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 796.28 examples/s]


In [15]:
first_half_dict, second_half_dict

(DatasetDict({
     train: Dataset({
         features: ['messages', 'task_id'],
         num_rows: 242
     })
     validation: Dataset({
         features: ['messages', 'task_id'],
         num_rows: 50
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['messages', 'task_id'],
         num_rows: 243
     })
     validation: Dataset({
         features: ['messages', 'task_id'],
         num_rows: 50
     })
 }))

In [16]:
bottom_10_dict = create_dataset_dict(bottom_10_with_data, True, 10)
top_10_dict = create_dataset_dict(top_10_with_data, False, 10)
bottom_30_dict = create_dataset_dict(bottom_30_with_data, True, 30)
top_30_dict = create_dataset_dict(top_30_with_data, False, 30)

In [18]:
top_50_ids = second_half_dict["train"]["task_id"]
top_30_ids = top_30_dict["train"]["task_id"]
for id in top_30_ids:
    assert id in top_50_ids

bottom_50_ids = first_half_dict["train"]["task_id"]
bottom_30_ids = bottom_30_dict["train"]["task_id"]
for id in bottom_30_ids:
    assert id in bottom_50_ids

In [19]:
save_ds_s3(first_half_dict, "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset_redo_0811/llama-3-instruct/low_50/dataset/")
save_ds_s3(second_half_dict, "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset_redo_0811/llama-3-instruct/high_50/dataset/")

Saving the dataset (1/1 shards): 100%|██████████| 50/50 [00:00<00:00, 1185.65 examples/s]


In [20]:
save_ds_s3(bottom_30_dict, "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset_redo_0811/llama-3-instruct/low_30/dataset/")
save_ds_s3(top_30_dict, "s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset_redo_0811/llama-3-instruct/high_30/dataset/")

Saving the dataset (1/1 shards): 100%|██████████| 30/30 [00:00<00:00, 620.20 examples/s]


# Train on datasets saved to s3

In [6]:
def start_train_job(dataset, lr, split="low50"):
    params = {
        "user_id": "meher.mankikar",
        "compute_type": "k8s",
        "model_checkpoint_path": "s3://scale-ml/models/llama-3/llama-3-8b-instruct-052124/",
        "base_model_name": "custom-8B",
        "data_path": dataset,
        "training_method": "sft",
        "training_hyperparams": {
            "learning_rate": lr,
            "num_train_epochs": 8,
            "max_length": 8192,
            "eval_steps": 0.1
        },
        "save_steps": 0.1,
        "save_strategy": "steps",
        "output_dir": f"s3://scale-ml/users/mehermankikar/precog/coding/finetuned_models/llama-3-8b-instruct-052124-larger-dataset_redo_0811/{split}_{lr}/model",
        "wandb_config": {
            "group": f"0811_nuggets_llama-3-8b-instruct_redo_{split}",
            "name": f"0811_nuggets_llama-3-8b-instruct_redo_{split}_lr{lr}"
        }  
    }
    res = requests.post(
        f"{HOST}/precog/v1/train_runs",
        data=json.dumps(params),
        auth=basic
    )
    return JSON(res.json())

In [28]:
start_train_job("s3://scale-ml/users/mehermankikar/nuggets_datasets/debugging_large_dataset_redo_0811/llama-3-instruct/combined_ds/dataset/",
                1e-5, "combined_ds")


<IPython.core.display.JSON object>

In [29]:
# From looking at the weights and biases chart, we pick checkpoint 60 as the checkpoint with a low validation loss
params = {
    "user_id": "meher.mankikar",
    "model_checkpoint_path": "s3://scale-ml/users/mehermankikar/precog/coding/finetuned_models/llama-3-8b-instruct-052124-larger-dataset_removing_long_tasks/combined_ds_try2_2e-6/model/model/checkpoints/checkpoint-168/",
    "base_model_name": "custom-8B",
    "output_dir": "s3://scale-ml/users/mehermankikar/precog/coding_evals/debugging-large-dataset/llama-3-instruct_removing-long-tasks/combined_ds_try2_2e-06_chkpt_168/",
    "eval_config": [
        {
            "benchmark": "humaneval",
            "harness": "bigcode",
            "hyperparams": {
                "max_length_generation": 1024,
                "precision": "bf16",
                "do_sample": False
            }
        }
    ]
}
res = requests.post(
    f"{HOST}/precog/v1/eval_runs",
    data=json.dumps(params),
    auth=basic
)
JSON(res.json())


<IPython.core.display.JSON object>